# 03 - Bronze to Silver Load
## FoodLens Databricks Pipeline
### Purpose: Transform Bronze Delta Tables → Cleanse → Validate → Write Silver Delta Table

In [0]:
%run "./Util"

## Step 1: Load Bronze Tables

In [0]:
chicago_df = spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_CHICAGO_TABLE}")
dallas_df  = spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_DALLAS_TABLE}")

print(f"Chicago Bronze rows: {chicago_df.count():,}")
print(f"Dallas Bronze rows : {dallas_df.count():,}")

Chicago Bronze rows: 308,161
Dallas Bronze rows : 78,984


## Step 2: Preview Bronze Data

In [0]:
display(chicago_df.limit(10))

Inspection_ID,DBA_Name,AKA_Name,License_#,Facility_Type,Risk,Address,City,State,Zip,Inspection_Date,Inspection_Type,Results,Violations,Latitude,Longitude,Location,_batch_id,_source_city,_ingested_at,_bronze_loaded_at
2633502,O & M QUICK MART,O & M QUICK MART,3073369,Grocery Store,Risk 3 (Low),8625 S STONY ISLAND AVE,CHICAGO,IL,60617,04/03/2026,License Re-Inspection,Pass,"38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: 6-202.15 OBSERVED A 1/4 INCH GAP AT THE BOTTOM OF THE FRONT ENTRANCE DOOR. INSTRUCTED TO MAKE SAID DOOR TIGHT FITTING AND RODENT PROOFED. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: 6-201.11 OBSERVED THE REAR AREA HAS DRY WALL WHICH IS AN ABSORBENT SURFACE. INSTRUCTED TO FINISH THE WALL SO THAT THE WALL SURFACE IS SMOOTH AND EASILY CLEANABLE. -201.13 OBSERVED THE REAR STORAGE AREA AND THE TOILET ROOM WALL'S ARE NOT COVED. INSTRUCTED TO PROVIDE COVING TO MAKE THE SURFACE SMOOTH AND EASILY CLEANABLE.",41.73793841751,-87.58507814194,"(41.737938417513675, -87.58507814194496)",BATCH_20260421_201905,Chicago,2026-04-21T20:19:05.039Z,2026-04-21T20:19:17.706Z
2633490,TAMALERIA LUPITA 3 & CO.,TAMALERIA LUPITA,3078163,Restaurant,Risk 1 (High),3858 S CALIFORNIA AVE,CHICAGO,IL,60632,04/03/2026,License,Pass w/ Conditions,"5. PROCEDURES FOR RESPONDING TO VOMITING AND DIARRHEAL EVENTS - Comments: OBSERVED ALL NECESSARY ITEMS NOT ON PREMISES FOR COMPLYING WITH PROCEDURE AVAILABLE ON SITE WHEN EMPLOYEES RESPOND TO VOMITTING OR DIARRHEAL EVENTS. INSTRUCTED TO MAINTAIN ON PREMISES ALL NECESSARY ITEMS FOR COMPLYING WITH WRITTEN PROCEDURE WHEN EMPLOYEES RESPOND TO VOMITTING OR DIARRHEAL EVENTS. PRIORITY FOUNDATION VIOLATION | 38. INSECTS, RODENTS, & ANIMALS NOT PRESENT - Comments: OBSERVED OPENINGS AT BASE OF FRONT ENTRANCE OUTER DOOR. INSTRUCTED TO SEAL ALL OUTER OPENINGS TO PREVENT PEST ENTRANCE. | 44. UTENSILS, EQUIPMENT & LINENS: PROPERLY STORED, DRIED, & HANDLED - Comments: OBSERVED NO RIGHT SPLASH GUARD INSTALLED ON THE RIGHT SIDE OF DRAIN BOARD OF SANITIZER COMPARTMENT OF THREE COMPARTMENT SINK NEXT TO UTILITY SINK. INSTRUCTED TO INSTALL RIGHT SPLASH GUARD ON THE RIGHTSIDE OF DRAIN BOARD OF SANITIZER COMPARTMENT OF THREE COMPARTMENT SINK NEXT TO UTILITY SINK TO PREVENT CONTAMINATION OF SANITIZED UTENSILS AND EQUIPMENT. | 47. FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: OBSERVED PLASTIC FILM ON TOP AND BOTTOM OF PREP TABLE CLOSE TO REAR EXIT. INSTRUCTED TO REMOVE PLASTIC FILM ON TOP AND BOTTOM OF PREP TABLE CLOSE TO REAR EXIT, NON FOOD CONTACT SURFACES SHALL BE CONSTRUCTED OF A CORROSION-RESISTANT, NONABSORBENT AND SMOOTH MATERIAL. | 51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICES - Comments: OBSERVED LEAKING FROM DRAINPIPE UNDERNEATH RINSE COMPARTMENT OF THREE COMPARTMENT SINK. INSTRUCTED TO REPAIR LEAKING DRAINPIPE UNDERNEATH THREE COMPARTMENT SINK AND MAINTAIN. | 51. PLUMBING INSTALLED; PROPER BACKFLOW DEVICES - Comments: UNABLE TO LOCATE BACKFLOW PREVENTION DEVICE ON UTILITY SINK. INSTRUCTED TO INSTALL BACKFLOW PREVENTION DEVICE ON UTILITY SINK SO THAT IT SHALL BE LOCATED SO THAT IT MAY BE SERVICED AND MAINTAINED. | 55. PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN - Comments: OBSERVED DAMAGED AND MISSING FLOOR TILES THROUGHOUT PREP/DISHWASHING AREA. INSTRUCTED TO SMOOTHLY REPAIR OR REPLACE ALL DAMAGED OR MISSING FLOOR TILES IN FACILITY FOR EASE OF CLEANABILITY. | 56. ADEQUATE VENTILATION & LIGHTING; DESIGNATED AREAS USED - Comments: OBSERVED NO MECHANICAL VENTILATION INSTALLED IN ALL GENDER WASHROOM. INSTRUCTED TO INSTALL MECHANICAL VENTILATION OF SUFFICIENT CAPACITY IN ALL GENDER WASHROOM.",41.82283838377,-87.69477006147,"(41.82283838377082, -87.6947700614686)",BATCH_20260421_201905,Chicago,2026-04-21T20:19:05.039Z,2026-04-21T20:19:17.706Z
2633442,KENTUCKY FRIED CHICKEN #5286,KFC,2523424,Restaurant,Risk 1 (High),5852 S WESTERN AVE,CHICAGO,IL,60636,04/02/2026,Complaint Re-Inspection,Pass,null,41.7867880859,-87.68398488287,"(41.7867880859006

In [0]:
display(dallas_df.limit(10))

Restaurant_Name,Inspection_Type,Inspection_Date,Inspection_Score,Street_Number,Street_Name,Street_Direction,Street_Type,Street_Unit,Street_Address,Zip_Code,Violation_Description_-_1,Violation_Points_-_1,Violation_Detail_-_1,Violation_Memo_-_1,Violation_Description_-_2,Violation_Points_-_2,Violation_Detail_-_2,Violation_Memo_-_2,Violation_Description_-_3,Violation_Points_-_3,Violation_Detail_-_3,Violation_Memo_-_3,Violation_Description_-_4,Violation_Points_-_4,Violation_Detail_-_4,Violation_Memo_-_4,Violation_Description_-_5,Violation_Points_-_5,Violation_Detail_-_5,Violation_Memo_-_5,Violation_Description_-_6,Violation_Points_-_6,Violation_Detail_-_6,Violation_Memo_-_6,Violation_Description_-_7,Violation_Points_-_7,Violation_Detail_-_7,Violation_Memo_-_7,Violation_Description_-_8,Violation_Points_-_8,Violation_Detail_-_8,Violation_Memo_-_8,Violation_Description_-_9,Violation_Points_-_9,Violation_Detail_-_9,Violation_Memo_-_9,Violation_Description_-_10,Violation_Points_-_10,Violation_Detail_-_10,Violation_Memo_-_10,Violation_Description_-_11,Violation_Points_-_11,Violation_Detail_-_11,Violation_Memo_-_11,Violation_Description_-_12,Violation_Points_-_12,Violation_Detail_-_12,Violation_Memo_-_12,Violation_Description_-_13,Violation_Points_-_13,Violation_Detail_-_13,Violation_Memo_-_13,Violation_Description_-_14,Violation_Points_-_14,Violation_Detail_-_14,Violation_Memo_-_14,Violation_Description_-_15,Violation_Points_-_15,Violation_Detail_-_15,Violation_Memo_-_15,Violation_Description_-_16,Violation_Points_-_16,Violation_Detail_-_16,Violation_Memo_-_16,Violation_Description_-_17,Violation_Points_-_17,Violation_Detail_-_17,Violation_Memo_-_17,Violation_Description_-_18,Violation_Points_-_18,Violation_Detail_-_18,Violation_Memo_-_18,Violation_Description_-_19,Violation_Points_-_19,Violation_Detail_-_19,Violation_Memo_-_19,Violation_Description_-_20,Violation_Points_-_20,Violation_Detail_-_20,Violation__Memo_-_20,Violation_Description_-_21,Violation_Points_-_21,Violation_Detail_-_21,Violation_Memo_-_21,Violation_Description_-_22,Violation_Points_-_22,Violation_Detail_-_22,Violation_Memo_-_22,Violation_Description_-_23,Violation_Points_-_23,Violation_Detail_-_23,Violation_Memo_-_23,Violation_Description_-_24,Violation_Points_-_24,Violation_Detail_-_24,Violation_Memo_-_24,Violation_Description_-_25,Violation_Points_-_25,Violation_Detail_-_25,Violation_Memo_-_25,Inspection_Month,Inspection_Year,Lat_Long_Location,_batch_id,_source_city,_ingested_at,_bronze_loaded_at
TIENDA Y RESTAURANTE LA CAMPIñA SALvadorena,Routine,10/03/2016,66,10818,DENNIS,null,RD,102,10818 DENNIS RD 102,75229,"*01 Cooling -- within 2 hours, 135-70øF",3,"228.75 Food. Time and temperature control. (d) Cooling. (1) Cooked temperature/time controlled for safety food shall be cooled: (A) within two hours, from 57 degrees Celsius (135 degrees Fahrenheit) to 21 degrees C (70 degrees Fahrenheit); and",null,"*10 Chlorine sanitizer concentration, minimum temp.",3,"228.111 Equipment, Utensils, and Linens. Equipment, maintenance and operation. (n) Manual and mechanical warewashing equipment, chemical sanitization - temperature, pH, concentration, and hardness. A chemical sanitizer used in a sanitizing solution for a manual or mechanical operation at contact times specified in õ228.118(3) of this title shall meet the criteria in õ228.206(a) of this title, shall be used in accordance with the EPA-approved manufacturer's label use instructions, and shall be used as follows: (1) a chlorine solution shall have a minimum temperature based on the concentration and pH of the solution as listed in the following chart: Figure: 25 TAC õ228.111(n)(1)",null,*31 Handwashing lavatory - accessible,2,"228.149 Water, Plumbing, and Waste. Plumbing, operation and maintenance. (a) Using a handwashing facility. (1) A handwashing facility shall be maintained so that it is accessible at all times for employee use.",null,*21 RFSM - Not On Site,2,"Sec. 17-2.2(c)(1)(D) (c) Registered food service manage

## Step 3: Imports and Silver Schema Definition
Define the 26 standard columns for the unified Silver table.
One row per violation per inspection across both cities.

In [0]:
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.types import IntegerType, DoubleType, StringType
from functools import reduce

SILVER_COLS = [
    "inspection_id",
    "source_city",
    "restaurant_id",
    "dba_name",
    "aka_name",
    "license_num",
    "facility_type",
    "risk_category",
    "address",
    "change_hash",
    "zip_code",
    "inspection_date",
    "inspection_type",
    "inspection_result",
    "violation_score",
    "pass_fail_flag",
    "state",
    "latitude",
    "longitude",
    "violation_id",
    "violation_code",
    "violation_description",
    "violation_points",
    "violation_detail",
    "inspector_comments",
    "is_critical_flag",
]

print(f"Silver columns defined: {len(SILVER_COLS)}")

Silver columns defined: 26


## Step 4: Chicago Transformation
Steps:
1. Rename and cast all inspection columns
2. Derive violation_score from results text (Pass=90, Fail=70 etc.)
3. Explode violations string into one row per violation
4. Parse violation_code, description, comments via regex
5. Derive is_critical_flag from description keywords
6. Generate restaurant_id and inspection_id via MD5 hash

In [0]:
chi_silver = (
    chicago_df
    .withColumn("inspection_id",     F.trim(F.col("inspection_id")))
    .withColumn("source_city",       F.lit("CHICAGO"))
    .withColumn("dba_name",          F.upper(F.trim(F.col("dba_name"))))
    .withColumn("aka_name",          F.when(F.col("aka_name").isNull(),
                                            F.col("dba_name")).otherwise(F.upper(F.trim(F.col("aka_name")))))
    .withColumn("license_num",       F.trim(F.col("license_#").cast(IntegerType())))
    .withColumn("facility_type",     F.upper(F.trim(F.col("facility_type"))))
    .withColumn("risk_category",     F.trim(F.col("risk")))
    .withColumn("address",           F.upper(F.trim(F.col("address"))))
    .withColumn("zip_code",          F.trim(F.col("zip")))
    .withColumn("restaurant_id",     F.md5(F.concat_ws("|",
        F.upper(F.trim(F.col("dba_name").cast(StringType()))),
        F.upper(F.trim(F.col("facility_type").cast(StringType()))),
        F.upper(F.trim(F.col("address").cast(StringType()))),
        F.trim(F.col("zip_code").cast(StringType())))))
    .withColumn("change_hash",
        F.md5(F.coalesce(F.col("aka_name"), F.lit(""))))
    .withColumn("inspection_date",   F.to_date(F.col("inspection_date"), "MM/dd/yyyy"))
    .withColumn("inspection_type",   F.upper(F.trim(F.col("inspection_type"))))
    .withColumn("inspection_result", F.upper(F.trim(F.col("results"))))
    .withColumn("violation_score",
        F.when(F.upper(F.trim(F.col("results"))) == "PASS",               F.lit(90))
         .when(F.upper(F.trim(F.col("results"))) == "PASS W/ CONDITIONS", F.lit(80))
         .when(F.upper(F.trim(F.col("results"))) == "FAIL",               F.lit(70))
         .when(F.upper(F.trim(F.col("results"))) == "NO ENTRY",           F.lit(0))
         .otherwise(F.lit(None).cast(IntegerType())))
    .withColumn("pass_fail_flag",
        F.when(F.upper(F.trim(F.col("results"))).isin("PASS", "PASS W/ CONDITIONS"), "PASS")
         .when(F.upper(F.trim(F.col("results"))) == "FAIL", "FAIL")
         .otherwise("OTHER"))
    .withColumn("state",             F.upper(F.trim(F.col("state"))))
    .withColumn("latitude",          F.col("latitude").cast(DoubleType()))
    .withColumn("longitude",         F.col("longitude").cast(DoubleType()))
    .withColumn("violation_points",  F.lit(None).cast(IntegerType()))
    .withColumn("violation_detail",  F.lit(None).cast(StringType()))
    .withColumn("viol_entry",
        F.explode_outer(F.split(F.col("violations"), r"\s*\|\s*")))
    .filter(
        F.col("viol_entry").isNotNull() &
        (F.trim(F.col("viol_entry")) != ""))
    .withColumn("violation_code",
        F.regexp_extract("viol_entry", r"^(\d+)\.", 1))
    .withColumn("violation_description",
        F.trim(F.regexp_extract("viol_entry",
               r"^\d+\.\s*(.*?)\s*-\s*Comments\s*:", 1)))
    .withColumn("inspector_comments",
        F.trim(F.regexp_extract("viol_entry",
               r"-\s*Comments\s*:\s*(.*)", 1)))
    .withColumn("violation_id", F.md5(F.concat_ws("|",
            F.col("violation_code").cast(StringType()),
            F.col("violation_description").cast(StringType()),
            F.col("violation_detail").cast(StringType()))))
    .withColumn("is_critical_flag",
        F.when(
            F.upper(F.col("violation_description")).rlike(r"CRITICAL|URGENT") |
            F.upper(F.col("inspector_comments")).rlike(r"CRITICAL|URGENT"),
            F.lit("Y")
        ).otherwise(F.lit("N")))
    .filter(
        F.col("violation_code").isNotNull() &
        (F.col("violation_code") != ""))
    .select(SILVER_COLS)
)

print(f"Chicago silver : {chi_silver.count():,} rows")
display(chi_silver.limit(10))

Chicago silver : 1,001,388 rows


inspection_id,source_city,restaurant_id,dba_name,aka_name,license_num,facility_type,risk_category,address,change_hash,zip_code,inspection_date,inspection_type,inspection_result,violation_score,pass_fail_flag,state,latitude,longitude,violation_id,violation_code,violation_description,violation_points,violation_detail,inspector_comments,is_critical_flag
2633502,CHICAGO,c6c537a04b7ba681f9b9d33d05c6ca7c,O & M QUICK MART,O & M QUICK MART,3073369,GROCERY STORE,Risk 3 (Low),8625 S STONY ISLAND AVE,effcc43a33358cb5305069a79ed635b1,60617,2026-04-03,LICENSE RE-INSPECTION,PASS,90,PASS,IL,41.73793841751,-87.58507814194,30f30744fd4b0a65679b7f1a57cde069,38,"INSECTS, RODENTS, & ANIMALS NOT PRESENT",null,null,6-202.15 OBSERVED A 1/4 INCH GAP AT THE BOTTOM OF THE FRONT ENTRANCE DOOR. INSTRUCTED TO MAKE SAID DOOR TIGHT FITTING AND RODENT PROOFED.,N
2633502,CHICAGO,c6c537a04b7ba681f9b9d33d05c6ca7c,O & M QUICK MART,O & M QUICK MART,3073369,GROCERY STORE,Risk 3 (Low),8625 S STONY ISLAND AVE,effcc43a33358cb5305069a79ed635b1,60617,2026-04-03,LICENSE RE-INSPECTION,PASS,90,PASS,IL,41.73793841751,-87.58507814194,e1034e7086a177df105c319116140a0e,55,"PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN",null,null,6-201.11 OBSERVED THE REAR AREA HAS DRY WALL WHICH IS AN ABSORBENT SURFACE. INSTRUCTED TO FINISH THE WALL SO THAT THE WALL SURFACE IS SMOOTH AND EASILY CLEANABLE. -201.13 OBSERVED THE REAR STORAGE AREA AND THE TOILET ROOM WALL'S ARE NOT COVED. INSTRUCTED TO PROVIDE COVING TO MAKE THE SURFACE SMOOTH AND EASILY CLEANABLE.,N
2633490,CHICAGO,cb906525f515420f6284e01e5dbae186,TAMALERIA LUPITA 3 & CO.,TAMALERIA LUPITA,3078163,RESTAURANT,Risk 1 (High),3858 S CALIFORNIA AVE,12e49d2c47f818065dbda55d83d38c3c,60632,2026-04-03,LICENSE,PASS W/ CONDITIONS,80,PASS,IL,41.82283838377,-87.69477006147,c980278c2a09bf8419b27e55038b813e,5,PROCEDURES FOR RESPONDING TO VOMITING AND DIARRHEAL EVENTS,null,null,OBSERVED ALL NECESSARY ITEMS NOT ON PREMISES FOR COMPLYING WITH PROCEDURE AVAILABLE ON SITE WHEN EMPLOYEES RESPOND TO VOMITTING OR DIARRHEAL EVENTS. INSTRUCTED TO MAINTAIN ON PREMISES ALL NECESSARY ITEMS FOR COMPLYING WITH WRITTEN PROCEDURE WHEN EMPLOYEES RESPOND TO VOMITTING OR DIARRHEAL EVENTS. PRIORITY FOUNDATION VIOLATION,N
2633490,CHICAGO,cb906525f515420f6284e01e5dbae186,TAMALERIA LUPITA 3 & CO.,TAMALERIA LUPITA,3078163,RESTAURANT,Risk 1 (High),3858 S CALIFORNIA AVE,12e49d2c47f818065dbda55d83d38c3c,60632,2026-04-03,LICENSE,PASS W/ CONDITIONS,80,PASS,IL,41.82283838377,-87.69477006147,30f30744fd4b0a65679b7f1a57cde069,38,"INSECTS, RODENTS, & ANIMALS NOT PRESENT",null,null,OBSERVED OPENINGS AT BASE OF FRONT ENTRANCE OUTER DOOR. INSTRUCTED TO SEAL ALL OUTER OPENINGS TO PREVENT PEST ENTRANCE.,N
2633490,CHICAGO,cb906525f515420f6284e01e5dbae186,TAMALERIA LUPITA 3 & CO.,TAMALERIA LUPITA,3078163,RESTAURANT,Risk 1 (High),3858 S CALIFORNIA AVE,12e49d2c47f818065dbda55d83d38c3c,60632,2026-04-03,LICENSE,PASS W/ CONDITIONS,80,PASS,IL,41.82283838377,-87.69477006147,1df88589dab8733cf65b4ca98b5d5eeb,44,"UTENSILS, EQUIPMENT & LINENS: PROPERLY STORED, DRIED, & HANDLED",null,null,OBSERVED NO RIGHT SPLASH GUARD INSTALLED ON THE RIGHT SIDE OF DRAIN BOARD OF SANITIZER COMPARTMENT OF THREE COMPARTMENT SINK NEXT TO UTILITY SINK. INSTRUCTED TO INSTALL RIGHT SPLASH GUARD ON THE RIGHTSIDE OF DRAIN BOARD OF SANITIZER COMPARTMENT OF THREE COMPARTMENT SINK NEXT TO UTILITY SINK TO PREVENT CONTAMINATION OF SANITIZED UTENSILS AND EQUIPMENT.,N
2633490,CHICAGO,cb906525f515420f6284e01e5dbae186,TAMALERIA LUPITA 3 & CO.,TAMALERIA LUPITA,3078163,RESTAURANT,Risk 1 (High),3858 S CALIFORNIA AVE,12e49d2c47f818065dbda55d83d38c3c,60632,2026-04-03,LICENSE,PASS W/ CONDITIONS,80,PASS,IL,41.82283838377,-87.69477006147,25ea7c3560e553a1ff962f0aca9c38d6,47,"FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED",null,null,"OBSERVED PLASTIC FILM ON TOP AND BOTTOM OF PREP TABLE CLOSE TO REAR EXIT. INSTRUCTED TO REMOVE PLASTIC FILM ON TOP AND BOTTOM OF PREP TABLE CLOSE TO REAR EXIT, NON FOOD CON

## Step 5: Chicago DQX Validation
Drop rows that fail mandatory field checks.

In [0]:
class _DQCheck:
    def __init__(self, condition, msg):
        self.condition, self.msg = condition, msg

class _DQEngine:
    def apply_checks_and_split(self, df, checks):
        err = F.lit(None).cast("string")
        for c in reversed(checks):
            err = F.when(~c.condition, F.lit(c.msg)).otherwise(err)
        tagged = df.withColumn("_dq_error", err)
        return (tagged.filter(F.col("_dq_error").isNull()).drop("_dq_error"),
                tagged.filter(F.col("_dq_error").isNotNull()))

chi_good, chi_bad = _DQEngine().apply_checks_and_split(chi_silver, [
    _DQCheck(F.col("dba_name").isNotNull() & (F.trim(F.col("dba_name")) != ""),
             "dba_name is null/empty"),
    _DQCheck(F.col("inspection_date").isNotNull(),
             "inspection_date is null"),
    _DQCheck(F.col("inspection_type").isNotNull() & (F.trim(F.col("inspection_type")) != ""),
             "inspection_type is null/empty"),
    _DQCheck(F.col("zip_code").isNotNull() & F.col("zip_code").rlike(r"^\d{5}$"),
             "zip_code is null or invalid format"),
    _DQCheck(F.col("inspection_result").isNotNull() & (F.trim(F.col("inspection_result")) != ""),
             "inspection_result is null/empty"),
])

print(f"Chicago — good: {chi_good.count():,}  bad: {chi_bad.count():,}")
print("\nBad rows by rule:")
chi_bad.groupBy("_dq_error").count().orderBy(F.desc("count")).show(truncate=False)

Chicago — good: 1,001,339  bad: 49

Bad rows by rule:
+----------------------------------+-----+
|_dq_error                         |count|
+----------------------------------+-----+
|zip_code is null or invalid format|49   |
+----------------------------------+-----+



## Step 5b: Chicago Business Rules
Remove PASS inspections that have critical violations.

In [0]:
chi_good = chi_good.filter(
    ~((F.col("inspection_result") == "PASS") &
      (F.col("is_critical_flag")  == "Y"))
)

print(f"Chicago after business rules : {chi_good.count():,} rows")

Chicago after business rules : 1,001,293 rows


## Step 6: Dallas Transformation
Steps:
1. Normalize 25 wide violation columns
2. Standardize inspection columns
3. Derive facility_type from keyword matching on DBA name
4. Extract lat/long from string
5. Unpivot wide violation columns into one row per violation
6. Generate IDs via MD5 hash

In [0]:
for c in dallas_df.columns:
    clean = c.lower().replace("_-_", "_").replace("__", "_")
    if clean != c:
        dallas_df = dallas_df.withColumnRenamed(c, clean)

dal_base = (
    dallas_df
    .withColumn("source_city",       F.lit("DALLAS"))
    .withColumn("dba_name",          F.upper(F.trim(F.col("restaurant_name"))))
    .withColumn("aka_name",          F.upper(F.trim(F.col("dba_name"))))
    .withColumn("address",           F.upper(F.trim(F.col("street_address"))))
    .withColumn("zip_code",          F.trim(F.col("zip_code")))
    .withColumn("inspection_date",   F.to_date(F.col("inspection_date"), "MM/dd/yyyy"))
    .withColumn("inspection_type",   F.upper(F.trim(F.col("inspection_type"))))
    .withColumn("violation_score",   F.col("inspection_score").cast(IntegerType()))
    .withColumn("inspection_result",
        F.when(F.col("inspection_score").cast(IntegerType()) >= 70, "PASS")
         .otherwise("FAIL"))
    .withColumn("pass_fail_flag",
        F.when(F.col("inspection_score").cast(IntegerType()) >= 70, "PASS")
         .otherwise("FAIL"))
    .withColumn("state",             F.lit("TX"))
    .withColumn("latitude",
        F.regexp_extract(F.col("lat_long_location"), r"\(\s*(-?[\d.]+)\s*,", 1)
         .cast(DoubleType()))
    .withColumn("longitude",
        F.regexp_extract(F.col("lat_long_location"), r",\s*(-?[\d.]+)\s*\)", 1)
         .cast(DoubleType()))
    .withColumn("license_num",   F.lit(None).cast(IntegerType()))
    .withColumn("facility_type",
        F.when(F.lower(F.col("dba_name")).rlike("school"), "School")
        .when(F.lower(F.col("dba_name")).rlike("cafe|starbucks"), "Cafe")
        .when(F.lower(F.col("dba_name")).rlike("grocery"), "Grocery Store")
        .when(F.lower(F.col("dba_name")).rlike("restaurant|taqueria|olive garden|panera"), "Restaurant")
        .when(F.lower(F.col("dba_name")).rlike("food|catering|tortilleria|salad"), "Food Service")
        .when(F.lower(F.col("dba_name")).rlike("bakery"), "Bakery")
        .when(F.lower(F.col("dba_name")).rlike("golf"), "Recreational Facility")
        .when(F.lower(F.col("dba_name")).rlike("bar"), "Bar")
        .when(F.lower(F.col("dba_name")).rlike("children facility"), "Childrens Service Facility")
        .when(F.lower(F.col("dba_name")).rlike(
                "mcdonald|burger king|sonic|wingstop|jack in the box|golden chick|kfc|whataburger|popeyes|el pollo regio|williams fried chicken|fried chicken|subway|chipotle|chick-fil-a|pizza|taco|popeye|sushi|wendy|donut|chicken|panda express|ihop|burger|rice bowl"
            ), "Fast Food")
        .when(F.lower(F.col("dba_name")).rlike(
            "7-eleven|cvs pharmacy|7-11|dollar tree|trader joe|cvs|daily|dely"), "Retail / Convenience")
        .otherwise(F.col("dba_name")))
    .withColumn("restaurant_id",     F.md5(F.concat_ws("|",
        F.upper(F.trim(F.col("dba_name").cast(StringType()))),
        F.upper(F.trim(F.col("facility_type").cast(StringType()))),
        F.upper(F.trim(F.col("address").cast(StringType()))),
        F.trim(F.col("zip_code").cast(StringType())))))
    .withColumn("risk_category", F.lit("Unknown").cast(StringType()))
    .withColumn("change_hash",
        F.md5(F.coalesce(F.col("aka_name"), F.lit(""))))
    .withColumn("inspection_id",
        F.md5(F.concat_ws("|",
            F.upper(F.trim(F.col("dba_name"))),
            F.to_date(F.col("inspection_date"), "MM/dd/yyyy").cast(StringType()),
            F.upper(F.trim(F.col("address"))),
            F.trim(F.col("zip_code")))))
)

desc_cols = sorted(
    [c for c in dal_base.columns if c.startswith("violation_description_")],
    key=lambda x: int(x.split("_")[-1])
)
pts_cols  = sorted(
    [c for c in dal_base.columns if c.startswith("violation_points_")],
    key=lambda x: int(x.split("_")[-1])
)
det_cols  = sorted(
    [c for c in dal_base.columns if c.startswith("violation_detail_")],
    key=lambda x: int(x.split("_")[-1])
)
memo_cols = sorted(
    [c for c in dal_base.columns if c.startswith("violation_memo_")],
    key=lambda x: int(x.split("_")[-1])
)

print(f"Dallas violation slots found: {len(desc_cols)}")

inspection_cols = [
    "inspection_id", "source_city", "restaurant_id", "dba_name", "aka_name",
    "license_num", "facility_type", "risk_category", "address", "change_hash", "state",
    "zip_code", "inspection_date", "inspection_type", "inspection_result",
    "violation_score", "pass_fail_flag", "latitude", "longitude",
]

slot_dfs = []
for desc_c, pts_c, det_c, memo_c in zip(desc_cols, pts_cols, det_cols, memo_cols):
    slot_df = (
        dal_base
        .filter(F.col(desc_c).isNotNull() & (F.trim(F.col(desc_c)) != ""))
        .select(
            *inspection_cols,
            F.when(
                F.regexp_extract(F.col(desc_c), r"^\*\s*(\d+)", 1) != "",
                F.regexp_extract(F.col(desc_c), r"^\*\s*(\d+)", 1)
            ).otherwise(F.lit("UNK")).alias("violation_code"),
            F.when(
                F.regexp_extract(F.col(desc_c), r"^\*\s*\d+\s+(.*)", 1) != "",
                F.regexp_replace(
                    F.regexp_extract(F.col(desc_c), r"^\*\s*\d+\s+(.*)", 1),
                    "ø", "\u00b0")
            ).otherwise(
                F.regexp_replace(F.trim(F.col(desc_c)), "ø", "\u00b0")
            ).alias("violation_description"),
            F.col(pts_c).cast(IntegerType()).alias("violation_points"),
            F.regexp_replace(F.col(det_c),  "ø", "\u00b0").alias("violation_detail"),
            F.regexp_replace(F.col(memo_c), "ø", "\u00b0").alias("inspector_comments"),
            F.when(F.col(pts_c).cast(IntegerType()) == 3, F.lit("Y"))
             .otherwise(F.lit("N")).alias("is_critical_flag"),
        )
        .withColumn("violation_id", F.md5(F.concat_ws("|",
            F.col("violation_code").cast(StringType()),
            F.col("violation_description").cast(StringType()),
            F.col("violation_detail").cast(StringType()))))
        .select(SILVER_COLS)
    )
    slot_dfs.append(slot_df)

dal_silver = reduce(DataFrame.unionAll, slot_dfs)

print(f"Dallas silver : {dal_silver.count():,} rows")
display(dal_silver.limit(10))

Dallas violation slots found: 25
Dallas silver : 398,406 rows


inspection_id,source_city,restaurant_id,dba_name,aka_name,license_num,facility_type,risk_category,address,change_hash,zip_code,inspection_date,inspection_type,inspection_result,violation_score,pass_fail_flag,state,latitude,longitude,violation_id,violation_code,violation_description,violation_points,violation_detail,inspector_comments,is_critical_flag
93fa98e8e7d01b9047b8f34b7935aa67,DALLAS,3ae9a8bc6a1322088e7c2bed46c1f7a0,TIENDA Y RESTAURANTE LA CAMPIÑA SALVADORENA,TIENDA Y RESTAURANTE LA CAMPIÑA SALVADORENA,null,Restaurant,Unknown,10818 DENNIS RD 102,1e369c5603fc937246a0229b8aa95476,75229,2016-10-03,ROUTINE,FAIL,66,FAIL,TX,32.895847,-96.881391,ba98508ec3fbfdb441bb3b3818a08e65,01,"Cooling -- within 2 hours, 135-70°F",3,"228.75 Food. Time and temperature control. (d) Cooling. (1) Cooked temperature/time controlled for safety food shall be cooled: (A) within two hours, from 57 degrees Celsius (135 degrees Fahrenheit) to 21 degrees C (70 degrees Fahrenheit); and",null,Y
c0ba707f77c423d8fb2bb44fbf660c00,DALLAS,853102b98c652aefb8a04be5c72a7f3f,TORTILLERIA LA ESPIGA,TORTILLERIA LA ESPIGA,null,Food Service,Unknown,1328 N JIM MILLER RD #104,bf63e067a7eaea40d08300593e675ef5,75217,2016-10-03,ROUTINE,PASS,82,PASS,TX,32.73556,-96.700079,bbb3f7f9e8964245fc0526eaa14c589b,39,"In-use utensils, between-use storage. During pauses in food preparation or dispensing, food prep",1,"228.68 Food. Preventing Contamination From Equipment, Utensils, and Linens. (b) In-use utensils, between-use storage. During pauses in food preparation or dispensing, food preparation and dispensing utensils shall be stored: (2) in food that is not time/temperature controlled for safety (TCS) with their handles above the top of the food within containers or equipment that can be closed, such as bins of sugar, flour, or cinnamon;",cannot use water containers to keep utensils to reuse again,N
972280370817d3c407f3467fcd56382f,DALLAS,db4be7a688ebc2b3989f0f85039f6908,TMGM @ KEETON PARK GOLF COURSE,TMGM @ KEETON PARK GOLF COURSE,null,Recreational Facility,Unknown,2323 N JIM MILLER RD,656130b8f2948c74c817d9dcbb689472,75227,2016-10-03,ROUTINE,PASS,80,PASS,TX,32.756246,-96.701964,6b18e9b18531dbd3b5c97920a58fd915,12,A food employee shall comply with an exclusion or RESTRICTION,3,"228.35 Management and Personnel. Responsibilities and Reporting Symptoms and Diagnosis. (f) A food employee shall: (2) comply with a restriction as specified in õ228.36(4)(B), (5)(B), (6)(B), (7), (8)(B), or õ228.36 (8), (9), or (10) and comply with the provisions specified in õ228.37(4) - (10) of this title.",missing employee health policies,Y
60c8d6fb9a5990eebeaea59c1ff7ed06,DALLAS,04a1e723af0e4990305fe3192444dfe7,TORTAS Y TACOS EL RANCHITO #4,TORTAS Y TACOS EL RANCHITO #4,null,Fast Food,Unknown,5560 E GRAND AVE,ac5b3ad3da2b2d679330dd358d1e0c4f,75223,2016-10-03,ROUTINE,PASS,83,PASS,TX,32.793596,-96.747508,53813e2d6c7d137af1cc9df2886383be,03,Food products not maintained at 135°F or above,3,"228.75 Food. Time and temperature control. (f) Time/temperature controlled for safety food, hot and cold holding. (1) Except during preparation, cooking, or cooling, or when time is used as the public health control as specified in subsection (i) of this sectionin paragraphs (2) and (3) of this subsection, TCS food shall be maintained: (A) at 57 degrees Celsius (135 degrees Fahrenheit) or above, except that roasts cooked to a temperature and for a time specified in õ228.71(a)(2) of this title or reheated as specified in subsection õ228.73(e) of this title may be held at a temperature of 54 degrees Celsius (130 degrees Fahrenheit);",refried beans were at 130 degrees in some area and 125 in other areas ( stir product while on the hot holding unit.),Y
c281c414fb7b0316162ad584a6d608a5,DALLAS,b6c531134bec7ca80f3ff2417a286aff,ROYAL TAQUERIA,ROYAL TAQUERIA,null,Restaurant,Unknown,10818 DENNIS RD #103,32222aa0032d335d6799b3d133994a72,75229,2016-10-03,ROUTINE,PASS,81,PASS,TX,32.895847,-96.881391,4fd0d2c8773636394c947c26a33d118

## Step 7: Dallas DQX Validation
Drop rows that fail mandatory field checks and range validations.

In [0]:
dal_good, dal_bad = _DQEngine().apply_checks_and_split(dal_silver, [
    _DQCheck(F.col("dba_name").isNotNull() & (F.trim(F.col("dba_name")) != ""),
             "dba_name is null/empty"),
    _DQCheck(F.col("inspection_date").isNotNull(),
             "inspection_date is null"),
    _DQCheck(F.col("inspection_type").isNotNull() & (F.trim(F.col("inspection_type")) != ""),
             "inspection_type is null/empty"),
    _DQCheck(F.col("zip_code").isNotNull() & F.col("zip_code").rlike(r"^\d{5}$"),
             "zip_code is null or invalid format"),
    _DQCheck(
        F.col("violation_score").isNotNull() &
        (F.col("violation_score").cast(DoubleType()) >= 0) &
        (F.col("violation_score").cast(DoubleType()) <= 100),
        "violation_score out of range 0-100"),
])

print(f"Dallas  — good: {dal_good.count():,}  bad: {dal_bad.count():,}")
print("\nBad rows by rule:")
dal_bad.groupBy("_dq_error").count().orderBy(F.desc("count")).show(truncate=False)

Dallas  — good: 397,106  bad: 1,300

Bad rows by rule:
+----------------------------------+-----+
|_dq_error                         |count|
+----------------------------------+-----+
|zip_code is null or invalid format|1186 |
|dba_name is null/empty            |75   |
|violation_score out of range 0-100|39   |
+----------------------------------+-----+



## Step 8: Dallas Business Rules
Rule 1 — PASS cannot have critical violations
Rule 2 — Score >= 90 cannot have more than 3 violations

In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("inspection_id")

dal_good = (
    dal_good
    .filter(~((F.col("inspection_result") == "PASS") &
               (F.col("is_critical_flag")  == "Y")))
    .withColumn("_viol_cnt", F.count("*").over(w))
    .filter(~((F.col("violation_score") >= 90) &
               (F.col("_viol_cnt") > 3)))
    .drop("_viol_cnt")
)

print(f"Dallas final : {dal_good.count():,} rows")

Dallas final : 240,800 rows


## Step 9: Deduplicate Violations
Remove duplicate violations per inspection for both cities.

In [0]:
chi_good = chi_good.dropDuplicates(["inspection_id", "inspection_date", "violation_code", "violation_description"])
dal_good = dal_good.dropDuplicates(["inspection_id", "inspection_date", "violation_code", "violation_description"])

print(f"Chicago after dedup : {chi_good.count():,}")
print(f"Dallas  after dedup : {dal_good.count():,}")

Chicago after dedup : 939,077
Dallas  after dedup : 240,211


## Step 10: Union Chicago + Dallas into Single Silver Table

In [0]:
silver = (
    chi_good
    .unionByName(dal_good)
    .withColumn("_silver_loaded_at", F.current_timestamp())
)

print(f"Unified silver : {silver.count():,} rows")
print("\nSource city breakdown:")
silver.groupBy("source_city").agg(
    F.count("*").alias("violation_rows"),
    F.countDistinct("inspection_id","inspection_date","restaurant_id","violation_score").alias("unique_inspections")
).show()
print("\nSilver schema:")
silver.printSchema()

Unified silver : 1,179,288 rows

Source city breakdown:
+-----------+--------------+------------------+
|source_city|violation_rows|unique_inspections|
+-----------+--------------+------------------+
|     DALLAS|        240211|             57738|
|    CHICAGO|        939077|            221608|
+-----------+--------------+------------------+


Silver schema:
root
 |-- inspection_id: string (nullable = true)
 |-- source_city: string (nullable = false)
 |-- restaurant_id: string (nullable = false)
 |-- dba_name: string (nullable = true)
 |-- aka_name: string (nullable = true)
 |-- license_num: long (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- risk_category: string (nullable = true)
 |-- address: string (nullable = true)
 |-- change_hash: string (nullable = false)
 |-- zip_code: string (nullable = true)
 |-- inspection_date: date (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- inspection_result: string (nullable = true)
 |-- violation_score: in

## Step 11: Write Silver Delta Table

In [0]:
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")
)
print(f"✅ Written to: {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

✅ Written to: foodlens.silver_zone.silver_table


## Step 12: Integrity Checks
Verify zero bad rows made it through to Silver.

In [0]:
st = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

insp_total   = st.select("inspection_id").distinct().count()
insp_w_viols = st.filter(
    F.col("violation_code").isNotNull() & (F.col("violation_code") != "")
).select("inspection_id").distinct().count()
pass_critical  = st.filter(
    (F.col("inspection_result") == "PASS") & (F.col("is_critical_flag") == "Y") & (F.col("source_city") == "CHICAGO")
).count()
over_100 = st.filter(
    (F.col("source_city") == "DALLAS") & (F.col("violation_score") > 100)
).count()
dal_score_fail = (
    st.filter((F.col("source_city") == "DALLAS") & (F.col("violation_score") >= 90))
    .groupBy("inspection_id").agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 3).count()
)
bad_zips = st.filter(~F.col("zip_code").rlike(r"^\d{5}$")).count()

print(f"Silver table rows   : {st.count():,}  cols: {len(st.columns)}")
print(f"Inspections w/ viols: {insp_w_viols:,}")
print(f"PASS + critical viols: {pass_critical:,}")
print(f"Dallas score > 100  : {over_100:,}")
print(f"Dallas score>=90 >3v: {dal_score_fail:,}")
print(f"Invalid zip codes   : {bad_zips:,}")

all_ok = all([
    pass_critical  == 0,
    over_100       == 0,
    dal_score_fail == 0,
    bad_zips       == 0,
])
print(f"\n{'✅ ALL CHECKS PASSED' if all_ok else '❌ SOME CHECKS FAILED'}")

Silver table rows   : 1,179,288  cols: 27
Inspections w/ viols: 279,398
PASS + critical viols: 0
Dallas score > 100  : 0
Dallas score>=90 >3v: 0
Invalid zip codes   : 0

✅ ALL CHECKS PASSED


In [0]:
# ──────────────────────────── CELL 13 : Summary ────────────────────────────────────────
print("\n" + "=" * 55)
print("SILVER LAYER SUMMARY")
print("=" * 55)
print(f"Table : food_inspections.silver.silver_table")
print(f"Rows  : {st.count():,}   Cols: {len(st.columns)}")
print("\nRows by city:")
st.groupBy("source_city").agg(
    F.count("*").alias("violation_rows"),
    F.countDistinct("inspection_id","inspection_date","restaurant_id","violation_score").alias("unique_inspections")
).show()
print("Score distribution:")
st.groupBy("source_city", "inspection_result", "violation_score").count().orderBy(
    "source_city", "violation_score"
).show(30)
print("\nNext → 03_gold_dimensions.py")
print("Read  → food_inspections.silver.silver_table")


SILVER LAYER SUMMARY
Table : food_inspections.silver.silver_table
Rows  : 1,179,288   Cols: 27

Rows by city:
+-----------+--------------+------------------+
|source_city|violation_rows|unique_inspections|
+-----------+--------------+------------------+
|     DALLAS|        240211|             57738|
|    CHICAGO|        939077|            221608|
+-----------+--------------+------------------+

Score distribution:
+-----------+------------------+---------------+------+
|source_city| inspection_result|violation_score| count|
+-----------+------------------+---------------+------+
|    CHICAGO|         NOT READY|           NULL|   426|
|    CHICAGO|   OUT OF BUSINESS|           NULL|   139|
|    CHICAGO|          NO ENTRY|              0|  4091|
|    CHICAGO|              FAIL|             70|342901|
|    CHICAGO|PASS W/ CONDITIONS|             80|226274|
|    CHICAGO|              PASS|             90|365246|
|     DALLAS|              FAIL|             44|    25|
|     DALLAS|       

## Step 13: Silver Layer Summary

In [0]:
print("=" * 55)
print("SILVER LAYER SUMMARY")
print("=" * 55)
print(f"Table : {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")
print(f"Rows  : {st.count():,}   Cols: {len(st.columns)}")
print("\nRows by city:")
st.groupBy("source_city").agg(
    F.count("*").alias("violation_rows"),
    F.countDistinct("inspection_id","inspection_date","restaurant_id","violation_score").alias("unique_inspections")
).show()
print("\nScore distribution:")
st.groupBy("source_city", "inspection_result", "violation_score").count().orderBy(
    "source_city", "violation_score"
).show(30)
print("\n✅ Silver complete! Next → 04_Silver_to_Gold_Load")

SILVER LAYER SUMMARY
Table : foodlens.silver_zone.silver_table
Rows  : 1,179,288   Cols: 27

Rows by city:
+-----------+--------------+------------------+
|source_city|violation_rows|unique_inspections|
+-----------+--------------+------------------+
|     DALLAS|        240211|             57738|
|    CHICAGO|        939077|            221608|
+-----------+--------------+------------------+


Score distribution:
+-----------+------------------+---------------+------+
|source_city| inspection_result|violation_score| count|
+-----------+------------------+---------------+------+
|    CHICAGO|         NOT READY|           NULL|   426|
|    CHICAGO|   OUT OF BUSINESS|           NULL|   139|
|    CHICAGO|          NO ENTRY|              0|  4091|
|    CHICAGO|              FAIL|             70|342901|
|    CHICAGO|PASS W/ CONDITIONS|             80|226274|
|    CHICAGO|              PASS|             90|365246|
|     DALLAS|              FAIL|             44|    25|
|     DALLAS|          